<a href="https://colab.research.google.com/github/amishram25/Cloud_lab/blob/main/Assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install mrjob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.6/439.6 kB 15.0 MB/s eta 0:00:00


## Q1. Word Count

In [26]:
# (A) Manual
data = ["hadoop is fast", "hadoop is scalable"]

# MAP
mapped = []
for line in data:
    for w in line.split():
        mapped.append((w, 1))

# SHUFFLE
shuffle = {}
for k, v in mapped:
    shuffle.setdefault(k, []).append(v)

# REDUCE
result = {k: sum(v) for k, v in shuffle.items()}
print(result)


{'hadoop': 2, 'is': 2, 'fast': 1, 'scalable': 1}


In [27]:
# (B) mrjob
from mrjob.job import MRJob

class MRWordCount(MRJob):
    def mapper(self, _, line):
        for w in line.split():
            yield w, 1
    def reducer(self, k, vals):
        yield k, sum(vals)



## Q2. Character Count (ignore spaces)

In [34]:
# (A) Manual
text = "big data"

mapped = [(c, 1) for c in text if c != " "]

shuffle = {}
for k, v in mapped:
    shuffle.setdefault(k, []).append(v)

result = {k: sum(v) for k, v in shuffle.items()}
print(result)


{'b': 1, 'i': 1, 'g': 1, 'd': 1, 'a': 2, 't': 1}


In [35]:
# (B) mrjob
class MRCharCount(MRJob):
    def mapper(self, _, line):
        for c in line.replace(" ", ""):
            yield c, 1
    def reducer(self, k, vals):
        yield k, sum(vals)


## Q3. Average Word Length (Per Word)

In [36]:
# (A) Manual
text = "data science data big data"

mapped = [(w, (len(w), 1)) for w in text.split()]

shuffle = {}
for k, v in mapped:
    shuffle.setdefault(k, []).append(v)

avg_len = {}
for k, vals in shuffle.items():
    total = sum(x[0] for x in vals)
    count = sum(x[1] for x in vals)
    avg_len[k] = total / count

print(avg_len)


{'data': 4.0, 'science': 7.0, 'big': 3.0}


In [37]:
# (B) mrjob
class MRAvgWordLen(MRJob):
    def mapper(self, _, line):
        for w in line.split():
            yield w, (len(w), 1)
    def reducer(self, k, vals):
        total = count = 0
        for t, c in vals:
            total += t
            count += c
        yield k, total / count


## Q4. Global Average Word Length

In [38]:
# (A) Manual
text = "hadoop mapreduce spark"
words = text.split()

total_len = sum(len(w) for w in words)
count = len(words)

print("Global Average:", total_len / count)


Global Average: 6.666666666666667


In [39]:
# (B) mrjob
class MRGlobalAvg(MRJob):
    def mapper(self, _, line):
        for w in line.split():
            yield "avg", (len(w), 1)
    def reducer(self, k, vals):
        total = count = 0
        for t, c in vals:
            total += t
            count += c
        yield k, total / count


## Q5. Perform Q1–Q4 on File + Top 5 Words


In [40]:
# (A) Manual on file
from collections import Counter

with open("/content/shakespeare.txt", "r") as f:
    text = f.read().lower()

words = text.split()

# Q1: Word Count
wc = Counter(words)
print("Word Count (sample):", list(wc.items())[:10])


# Q2: Character Count
char_count = Counter(text.replace(" ", ""))
print("Character Count (sample):", list(char_count.items())[:10])

# Q3: Avg Word Length per word
stats = {}
for w in words:
    if w not in stats:
        stats[w] = [0, 0]
    stats[w][0] += len(w)
    stats[w][1] += 1

avg_word_len = {k: v[0]/v[1] for k, v in stats.items()}
print("Avg Word Length per word (sample):", list(avg_word_len.items())[:10])

# Q4: Global Avg Word Length
total_len = sum(len(w) for w in words)
print("Global Avg Length:", total_len / len(words))

# Top 5
print("Top 5 words:", wc.most_common(5))

Word Count (sample): [('\ufeffthe', 1), ('project', 320), ('gutenberg', 250), ('ebook', 13), ('of', 18126), ('the', 27729), ('complete', 243), ('works', 268), ('william', 311), ('shakespeare,', 2)]
Character Count (sample): [('\ufeff', 1), ('t', 331121), ('h', 237361), ('e', 448860), ('p', 58887), ('r', 238947), ('o', 315828), ('j', 4859), ('c', 88720), ('g', 68589)]
Avg Word Length per word (sample): [('\ufeffthe', 4.0), ('project', 7.0), ('gutenberg', 9.0), ('ebook', 5.0), ('of', 2.0), ('the', 3.0), ('complete', 8.0), ('works', 5.0), ('william', 7.0), ('shakespeare,', 12.0)]
Global Avg Length: 4.484685214825106
Top 5 words: [('the', 27729), ('and', 26099), ('i', 19540), ('to', 18762), ('of', 18126)]


In [41]:
# (B) mrjob for Word Count
class MRWordCountFile(MRJob):
    def mapper(self, _, line):
        for w in line.split():
            yield w.lower(), 1
    def reducer(self, k, vals):
        yield k, sum(vals)



## Q6. Average Marks per Student

In [43]:
# (A) Manual
data = ["A 80", "B 70", "A 90", "B 60", "A 100"]

mapped = [(x.split()[0], (int(x.split()[1]), 1)) for x in data]

shuffle = {}
for k, v in mapped:
    shuffle.setdefault(k, []).append(v)

avg_marks = {}
for k, vals in shuffle.items():
    total = sum(x[0] for x in vals)
    count = sum(x[1] for x in vals)
    avg_marks[k] = total / count

print(avg_marks)


{'A': 90.0, 'B': 65.0}


In [44]:
# (B) mrjob
class MRAvgMarks(MRJob):
    def mapper(self, _, line):
        name, score = line.split()
        yield name, (int(score), 1)
    def reducer(self, k, vals):
        total = count = 0
        for s, c in vals:
            total += s
            count += c
        yield k, total / count


## Q7. Average Salary per Department + Highest

In [45]:
# (A) Manual
data = ["HR 50000", "IT 70000", "HR 60000", "IT 80000"]

mapped = [(x.split()[0], (int(x.split()[1]), 1)) for x in data]

shuffle = {}
for k, v in mapped:
    shuffle.setdefault(k, []).append(v)

avg_salary = {}
for k, vals in shuffle.items():
    total = sum(x[0] for x in vals)
    count = sum(x[1] for x in vals)
    avg_salary[k] = total / count

print(avg_salary)
print("Highest Paid Dept:", max(avg_salary, key=avg_salary.get))


{'HR': 55000.0, 'IT': 75000.0}
Highest Paid Dept: IT


In [46]:
# (B) mrjob
class MRAvgSalary(MRJob):
    def mapper(self, _, line):
        dept, sal = line.split()
        yield dept, (int(sal), 1)
    def reducer(self, k, vals):
        total = count = 0
        for s, c in vals:
            total += s
            count += c
        yield k, total / count


## Q8. Average Temperature per Country

In [47]:
# (A) Manual
data = ["New York,38","London,29","Tokyo,35",
        "New York,32","Delhi,45","Ambala,35"]

mapped = [(x.split(",")[0], (int(x.split(",")[1]), 1)) for x in data]

shuffle = {}
for k, v in mapped:
    shuffle.setdefault(k, []).append(v)

avg_temp = {}
for k, vals in shuffle.items():
    total = sum(x[0] for x in vals)
    count = sum(x[1] for x in vals)
    avg_temp[k] = total / count

print(avg_temp)


{'New York': 35.0, 'London': 29.0, 'Tokyo': 35.0, 'Delhi': 45.0, 'Ambala': 35.0}


In [48]:
# (B) mrjob
class MRTemp(MRJob):
    def mapper(self, _, line):
        city, t = line.split(",")
        yield city, (int(t), 1)
    def reducer(self, k, vals):
        total = count = 0
        for t, c in vals:
            total += t
            count += c
        yield k, total / count


## Q9. Kaggle Dataset (Global Land Temperatures)


In [49]:
# (A) Manual
import csv

data = {}
with open("/content/GlobalLandTemperatures_GlobalLandTemperaturesByMajorCity.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["AverageTemperature"]:
            c = row["Country"]
            t = float(row["AverageTemperature"])
            if c not in data:
                data[c] = [0, 0]
            data[c][0] += t
            data[c][1] += 1

avg = {k: v[0]/v[1] for k, v in data.items()}
print(list(avg.items())[:10])


[("Côte D'Ivoire", 26.163737197524014), ('Ethiopia', 17.525072662298964), ('India', 25.809309238455356), ('Syria', 17.37058733360228), ('Egypt', 20.900406225165536), ('Turkey', 13.790997727026713), ('Iraq', 22.61434568965519), ('Thailand', 27.164733303650937), ('Brazil', 22.8475552351923), ('Germany', 8.916233733417545)]


In [50]:
# (B) mrjob
class MRCountryTemp(MRJob):
    def mapper(self, _, line):
        parts = line.split(',')
        if len(parts) >= 4 and parts[1]:
            country = parts[3]
            try:
                temp = float(parts[1])
                yield country, (temp, 1)
            except:
                pass
    def reducer(self, k, vals):
        total = count = 0
        for t, c in vals:
            total += t
            count += c
        yield k, total / count
